In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, root_mean_squared_error
from catboost import CatBoostRegressor, Pool


df = pd.read_json('All_Beauty.jsonl', lines=True)

df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)

df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['full_text'] = df['title'] + " " + df['text']

df['text_length'] = df['full_text'].apply(lambda x: len(x.split()))
df['verified_purchase'] = df['verified_purchase'].astype(int)
df['helpful_vote'] = df['helpful_vote'].fillna(0).astype(int)

X = df[['full_text', 'text_length', 'verified_purchase', 'helpful_vote']]
y = df['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y 
)

text_cols = ['full_text']

train_pool = Pool(data=X_train, label=y_train, text_features=text_cols)
test_pool = Pool(data=X_test, label=y_test, text_features=text_cols)

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.1,
    eval_metric='RMSE',
    task_type='CPU',              
    early_stopping_rounds=50,
    verbose=100                   
)

model.fit(train_pool, eval_set=test_pool)

raw_predictions = model.predict(X_test)

final_predictions = np.clip(np.round(raw_predictions), 1, 5).astype(int)